# Table 2: scripted and semi-scripted gait
Run from the repository root. This reproduces the reported Table 2 numbers using the original bout-level means and Welch tests: scripted bouts end before minute 2; semi-scripted bouts start after minute 3 and end before minute 40.

**Legacy date parsing is required for the reported values.** The original notebook used pandas/dateutil's default month-first interpretation of ambiguous day/month/year strings. We make that behaviour explicit with `format="mixed", dayfirst=False`. Consistent day-first parsing changes which bouts fall inside the windows and does not reproduce Table 2. Table 3 uses explicit day-first parsing. This is a manuscript reproducibility issue, not an endorsed date convention.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
OUT = Path("results")
OUT.mkdir(exist_ok=True)
from scipy.stats import ttest_ind

In [2]:
participants = pd.read_parquet('derived/participants.parquet', columns=['Participant','TestStart'])
participants = participants.loc[~participants.Participant.str.startswith('TOPAL')].copy()
bouts = pd.read_parquet('derived/bouts.parquet', columns=['Participant','BoutIndex','BoutStart','BoutEnd','BoutStepTCV'])
bouts = bouts.merge(participants, on='Participant', validate='many_to_one')

for col in ['BoutStart','BoutEnd']:
    bouts[col] = pd.to_datetime(bouts[col], format='mixed', dayfirst=False)

bouts['TestStart'] = pd.to_datetime(bouts.TestStart)
bouts['Group'] = np.where(bouts.Participant.str.contains('PD'), 'PwP', 'Control')
start = (bouts.BoutStart - bouts.TestStart).dt.total_seconds() / 60
end = (bouts.BoutEnd - bouts.TestStart).dt.total_seconds()/60
conditions = {'Scripted': end.lt(2), 'Semi-scripted': start.gt(3) & end.lt(40)}

rows=[]
for label,mask in conditions.items():
    subset=bouts.loc[mask]
    a=subset.loc[subset.Group.eq('PwP'),'BoutStepTCV'].dropna()
    b=subset.loc[subset.Group.eq('Control'),'BoutStepTCV'].dropna()
    rows.append({
        'Condition': label,
        'PwP': a.mean(),
        'Control': b.mean(),
        'Difference': round(round(a.mean(),3)-round(b.mean(),3),3),
        'Difference_unrounded':a.mean() - b.mean(),
        'p': ttest_ind(a,b,equal_var=False).pvalue,
        'PwP_bouts': len(a),
        'Control_bouts':len(b),
        'PwP_participants': subset.loc[subset.Group.eq('PwP') & subset.BoutStepTCV.notna(),'Participant'].nunique(),
        'Control_participants':subset.loc[subset.Group.eq('Control') & subset.BoutStepTCV.notna(),'Participant'].nunique()
    })
    
table2=pd.DataFrame(rows)
display(table2.round(3))
table2.to_csv(OUT/'table2.csv',index=False)

,Condition,PwP,Control,Difference,Difference_unrounded,p,PwP_bouts,Control_bouts,PwP_participants,Control_participants
0,Scripted,0.137,0.124,0.013,0.012,0.176,222,214,13,12
1,Semi-scripted,0.130,0.137,-0.007,-0.008,0.387,219,205,12,12
